In [ ]:
import pandas as pd
import numpy as np

import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
from IPython.display import display, clear_output

import sys
pythonScriptDir = "../scripts/python/"
sys.path.append(pythonScriptDir)

dataDir = "../data/"

In [ ]:
df = pd.read_csv("../data/raw/bach_chorales.dataset")
pitch_cols = [col for col in df.columns if 'pitch' in col.lower()]
bass_col = 'bass'

df[pitch_cols] = (df[pitch_cols] == "YES") # Converts YES to True and NO to False (hopefully)

In [ ]:
import sounddevice as sd
import threading
import time

from playback import get_audio

current_audio = None
playhead = 0
is_playing = False
sample_rate = 44100

def audio_callback(outdata, frames, time_info, status):
    global playhead, current_audio, is_playing

    if not is_playing or current_audio is None:
        outdata[:] = 0
        return

    end = playhead + frames
    chunk = current_audio[playhead:end]

    if len(chunk) < frames:
        outdata[:len(chunk), 0] = chunk
        outdata[len(chunk):, 0] = 0
        is_playing = False
    else:
        outdata[:, 0] = chunk

    playhead = end

stream = sd.OutputStream(
    samplerate=sample_rate,
    channels=1,
    callback=audio_callback
)
stream.start()

In [ ]:
from pitch_utils import inversion, INV_MAP, INV_LABELS

INV_COLORS = [
    "#1f77b4",  # root
    "#ff7f0e",  # 1st
    "#2ca02c",  # 2nd
    "#d62728",  # 3rd
    "#7f7f7f"   # non-chord
]
cmap = ListedColormap(INV_COLORS)

# Aggiungo una colonna che identifica il tipo di inversione utilizzata in ogni evento
df["inversion"] = df.apply(
    lambda r: inversion(r["chord_label"], r["bass"]), 
    axis=1
)
df["inv_code"] = df["inversion"].map(INV_MAP)

legend_patches = [
    mpatches.Patch(color=INV_COLORS[i], label=INV_LABELS[i])
    for i in range(len(INV_LABELS))
]

In [ ]:
def transition_matrix(sequence, labels):
    idx = {v: i for i, v in enumerate(labels)}
    mat = np.zeros((len(labels), len(labels)))

    for i in range(len(sequence) - 1):
        mat[idx[sequence[i]], idx[sequence[i+1]]] += 1

    row_sums = mat.sum(axis=1, keepdims=True)
    return np.divide(mat, row_sums, out=np.zeros_like(mat), where=row_sums != 0)


def plot_piece(piece_id):
    piece = df[df["choral_ID"] == piece_id]

    seq = piece["inv_code"].values
    meter = piece["meter"].values
    chords = piece["chord_label"].values
    x = np.arange(len(seq))

    fig = plt.figure(figsize=(15, 15))
    gs = gridspec.GridSpec(3, 3, height_ratios=[1.2, 1, 1.2])

    ax0 = fig.add_subplot(gs[0, :])
    ax1 = fig.add_subplot(gs[1, 0])
    ax2 = fig.add_subplot(gs[1, 1])
    ax3 = fig.add_subplot(gs[1, 2])

    # Graph 1
    meter = meter / np.max(meter)
    bars = ax0.bar(x, meter, width=0.9)

    for i, b in enumerate(bars):
        b.set_color(cmap(seq[i] / 4))

    ax0.set_xticks(x)
    ax0.set_xticklabels(chords, rotation=45, fontsize=7)
    ax0.set_ylim(0, 1)
    ax0.set_title("Inversions (color) + event weight (height)")
    ax0.legend(handles=legend_patches, loc="upper right")


    #################### TRANSITIONS MATRIXES ####################
    # Inversion
    inv_labels = list(range(5))
    mat1 = transition_matrix(seq, inv_labels)
    im1 = ax1.imshow(mat1, cmap="viridis")

    ax1.set_xticks(inv_labels)
    ax1.set_yticks(inv_labels)
    ax1.set_xticklabels(INV_LABELS, rotation=45)
    ax1.set_yticklabels(INV_LABELS)
    ax1.set_title("Inversion transitions")
    plt.colorbar(im1, ax=ax1)

    # Chords
    chord_labels = sorted(pd.unique(chords))
    mat2 = transition_matrix(chords, chord_labels)
    im2 = ax2.imshow(mat2, cmap="viridis")

    ax2.set_xticks(range(len(chord_labels)))
    ax2.set_yticks(range(len(chord_labels)))
    ax2.set_xticklabels(chord_labels, rotation=90)
    ax2.set_yticklabels(chord_labels)
    ax2.set_title("Chord transitions")
    plt.colorbar(im2, ax=ax2)

    # Chords (reduced)
    short_chords = np.array([c[:3] for c in chords])
    short_labels = sorted(pd.unique(short_chords))

    mat3 = transition_matrix(short_chords, short_labels)
    im3 = ax3.imshow(mat3, cmap="viridis")

    ax3.set_xticks(range(len(short_labels)))
    ax3.set_yticks(range(len(short_labels)))
    ax3.set_xticklabels(short_labels, rotation=90)
    ax3.set_yticklabels(short_labels)
    ax3.set_title("Chord transitions (reduced)")
    plt.colorbar(im3, ax=ax3)

    plt.tight_layout()

    with plot_out:
        plot_out.clear_output(wait=True)
        display(fig)
        plt.close(fig)


In [ ]:
# Leaderboards
def compute_counts(piece):
    seq = piece["inv_code"].values
    chords = piece["chord_label"].values

    N = len(seq)
    uniq = len(pd.unique(chords))

    non_chord_code = 4
    non_chord_count = int(np.sum(seq == non_chord_code))
    non_chord_ratio = non_chord_count / N if N > 0 else 0.0

    return {
        "n_events": int(N),
        "n_unique_chords": int(uniq),
        "non_chord_count": non_chord_count,
        "non_chord_ratio": float(non_chord_ratio)
    }

counts_df = (
    df.groupby("choral_ID", sort=False)
      .apply(compute_counts)
      .apply(pd.Series)
)

def plot_extremes_with_context(df_, col, k=10, title=""):
    sorted_df = df_.sort_values(col, ascending=True)

    least = sorted_df.head(k)
    most = sorted_df.tail(k)

    x_least = np.arange(k)
    x_most = np.arange(k) + k + 2 # space between extremes

    plt.figure(figsize=(12, 5))

    # --- least ---
    plt.bar(x_least, least[col].values)
    plt.xticks(list(x_least) + list(x_most),
               list(least.index.astype(str)) + list(most.index.astype(str)),
               rotation=45, ha="right")

    # --- most ---
    plt.bar(x_most, most[col].values)

    # divider line
    plt.axvline(k + 0.5, linestyle="--", alpha=0.4)

    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_extremes_with_context(counts_df, "n_events", 10, "Events (least vs most)")
plot_extremes_with_context(counts_df, "n_unique_chords", 10, "Unique chords (least vs most)")
plot_extremes_with_context(counts_df, "non_chord_count", 10, "Non-chord count (least vs most)")
plot_extremes_with_context(counts_df, "non_chord_ratio", 10, "Non-chord ratio (least vs most)")


In [ ]:
# WIDGETS
options = [("Select chorale", None)] + [
    (cid, cid) for cid in df["choral_ID"].unique()
]

dropdown = widgets.Dropdown(
    options=[("Select chorale", None)] + [(cid, cid) for cid in df["choral_ID"].unique()],
    value=None,
    description="Chorale:"
)


play_btn = widgets.Button(description="Play")
pause_btn = widgets.Button(description="Pause")
restart_btn = widgets.Button(description="Restart")

slider = widgets.IntSlider(description="Position", min=0, max=1, value=0)

# WIDGET'S FUNCTIONS
def load(change):
    global current_audio, playhead
    
    piece_id = change["new"]
    if piece_id is None:
        return
        
    chorale_df = df[df["choral_ID"] == piece_id]
    current_audio = get_audio(chorale_df, piece_id, pitch_cols, "bass", dataDir)
    
    playhead = 0
    slider.max = len(current_audio)

    plot_piece(piece_id)
dropdown.observe(load, names="value")

def play(_):
    global is_playing
    is_playing = True

def pause(_):
    global is_playing
    is_playing = False

def restart(_):
    global playhead
    playhead = 0

play_btn.on_click(play)
pause_btn.on_click(pause)
restart_btn.on_click(restart)

def seek(change):
    global playhead
    playhead = change["new"]
slider.observe(seek, names="value")

def sync():
    global playhead
    while True:
        if current_audio is not None:
            slider.value = min(playhead, slider.max)
        time.sleep(0.1)
threading.Thread(target=sync, daemon=True).start()

In [ ]:
display(dropdown)
display(widgets.HBox([play_btn, pause_btn, restart_btn]))
display(slider)

plot_out = widgets.Output() 
display(plot_out)
# keep medium to low volume for less distortion